# Fehlendes Tempo 30 an Schulen aus Mapillary-Verkehrszeichen → MapRoulette

Findet Tempo-30-Zeichen (DE:274-30) in der Nähe von Schulen und Kitas, wo keine Straße im
Umkreis von 50 m in OpenStreetMap eine 30 trägt — weder `maxspeed=30` noch eine zeitliche,
richtungs- oder fahrzeugbezogene Variante oder eine Zone. Daraus werden Aufgaben für die
MapRoulette-Challenge 52985. Läuft von Hand, wenn eine neue Runde fällig ist.

Aufgebaut wie die `maproulette_tasks.ipynb` der übrigen Kampagnen. Die Schritte liegen in
[`../campaign.py`](../campaign.py); was diese Kampagne ausmacht — was als gemapptes Tempo 30
zählt, Abstände, Aufgabentext — in [`../t30_campaign.py`](../t30_campaign.py).

**Ablauf**

| Schritt | Funktion |
| --- | --- |
| Verkehrszeichen von data.vizsim.de spiegeln | `campaign.sync_features` |
| Tempo-30-Zeichen einlesen | `campaign.load_features` |
| auf Deutschland beschneiden | `campaign.clip_to_boundary` |
| frische / kurzlebige Schilder aussortieren | `campaign.filter_stable_signs` |
| Abstand zur nächsten Tempo-30-Straße und zur nächsten Schule | `t30.filter_tempo30_ways`, `campaign.distance_to_nearest` |
| Zeichen derselben Stelle zusammenfassen | `campaign.dedupe_nearby` |
| bereits bestehende Aufgaben abziehen | `campaign.drop_near_existing_tasks`, `campaign.drop_known_task_names` |
| **neuestes** Mapillary-Bild je Zeichen holen | `campaign.newest_image_ids` |
| GeoJSON für MapRoulette schreiben | `campaign.build_maproulette_geojson` mit `t30.aufgabe` |
| in die Challenge hochladen (nur mit `CAMPAIGN_UPLOAD=1`) | `campaign.upload_tasks` |

Die Challenge holt ihre Aufgaben aus `maproulette_tasks_missing-t30_schools.geojson` auf
GitHub: Notebook laufen lassen, Datei committen und pushen, in MapRoulette „Rebuild tasks“.
Abschnitt 9 lädt stattdessen direkt über die API hoch — sobald MapRoulette Skripten wieder
einen Zugang gibt (siehe [README](README.md)).

## Einstellungen

In [ ]:
import json
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# campaign.py and t30_campaign.py live one level up. ".." works because Jupyter and
# nbconvert run the kernel in the notebook's directory - the same assumption as
# ../../output/ and ../utils/.
sys.path.insert(0, "..")
import campaign
import t30_campaign as t30

# Must match the date in ../0_prepare_osm_network.ipynb, which writes both extracts.
set_date = "261017"

ordner_zeichen = Path("../../output")
pfad_maxspeed = Path(f"../utils/processed_osm_files/processed_maxspeed_germany_{set_date}.parquet")
pfad_schulen = Path(f"../utils/processed_osm_files/processed_schools_germany_{set_date}.parquet")
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")

# Only signs last seen after this date ...
zuletzt_gesehen_nach = "2025-01-01"
# ... and seen over at least this many months - filters out temporary signs.
mindest_standzeit_monate = 12

challenge_id = 52985

# MapRoulette pulls the tasks from this file on GitHub (remoteGeoJson), so it keeps
# its name. Once uploads by script work again, section 9 replaces commit, push and
# "Rebuild tasks".
ziel_geojson = Path("maproulette_tasks_missing-t30_schools.geojson")

# From the environment, so a run on the server needs no edit to this notebook:
#   CAMPAIGN_SYNC=0    read ../../output/ as it is. On the server it is what the
#                      weekly run produced, and mirroring would overwrite its manifest.
#   CAMPAIGN_UPLOAD=1  upload the tasks to the challenge - refused by MapRoulette
#                      until it has tokens for scripts, see the README. Without it,
#                      section 9 only says how many it would upload.
sync = os.environ.get("CAMPAIGN_SYNC", "1") != "0"
hochladen = os.environ.get("CAMPAIGN_UPLOAD") == "1"

spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

print(f"Suchradius {t30.SUCHRADIUS_M:.0f} m, Schule bis {t30.SCHULRADIUS_M:.0f} m, zusammenfassen bis {t30.ZUSAMMENFASSEN_M:.0f} m")
print(f"spiegeln: {sync}, hochladen: {hochladen}")

## 1 · Verkehrszeichen holen und einlesen

In [ ]:
if sync:
    # Fetches only what is missing locally or newer on data.vizsim.de.
    campaign.sync_features(ordner_zeichen)

manifest = json.loads((ordner_zeichen / "ml-ts_metadata.json").read_text(encoding="utf-8"))
print("Datenstand:", manifest["processed_date"])
print("im letzten Lauf unvollständig:", manifest.get("last_run_incomplete") or "nichts")

In [ ]:
# Expected number of states from the tile cache; None without one.
erwartet = campaign.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

signs = campaign.load_features(
    ordner_zeichen,
    prefix=campaign.PREFIX_ZEICHEN,
    values=t30.ZEICHEN,
    columns=spalten,
    expect_files=erwartet,
)
# The parquets are cut along zoom-14 tiles, and border tiles reach abroad - where
# the OSM extract has no roads, so a sign there near a German school became a task.
signs = campaign.clip_to_boundary(signs, campaign.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5))
signs["last_seen_at"].str[:7].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Zeichen")

signs = campaign.filter_stable_signs(signs, zuletzt_gesehen_nach, mindest_standzeit_monate)

signs["last_seen_at"].str[:7].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1], title=f"zuletzt gesehen nach {zuletzt_gesehen_nach}, ≥ {mindest_standzeit_monate} Monate"
)
plt.tight_layout()

## 3 · Abstand zur nächsten Tempo-30-Straße und zur nächsten Schule

Als gemapptes Tempo 30 zählt, was die Runde 2025 zählte (`t30.filter_tempo30_ways`):
`maxspeed`, `:forward`, `:backward` oder `:hgv` gleich `30`, eine ihrer `:conditional`-Varianten
mit `30 @` oder `zone:maxspeed=DE:30` — auf den Straßentypen in `t30.STRASSENTYPEN`.

Schule heißt `amenity=school` oder `kindergarten`, als Punkt oder Fläche, höchstens
`t30.SCHULRADIUS_M` (450 m) entfernt — so weit reichten die 50-m- und 400-m-Puffer der Runde 2025.

In [ ]:
tempo30 = t30.filter_tempo30_ways(gpd.read_parquet(pfad_maxspeed))
signs["dist_t30_m"] = campaign.distance_to_nearest(signs, tempo30, t30.SUCHRADIUS_M)
del tempo30

schulen = t30.read_schools(pfad_schulen)
signs["dist_schule_m"] = campaign.distance_to_nearest(signs, schulen, t30.SCHULRADIUS_M)
del schulen

ohne_t30 = signs["dist_t30_m"] == float("inf")
an_schule = signs["dist_schule_m"] < float("inf")
print(f"ohne Tempo 30 in {t30.SUCHRADIUS_M:.0f} m: {ohne_t30.sum()}, davon an einer Schule: {(ohne_t30 & an_schule).sum()}")

## 4 · Kandidaten

Zeichen, die höchstens `t30.ZUSAMMENFASSEN_M` (30 m) auseinander stehen, werden zu einer
Aufgabe; es bleibt das zuletzt gesehene. Die Runde 2025 hatte das schon berechnet, aber nicht
benutzt — die Datei enthielt alle Zeichen.

In [ ]:
kandidaten = signs[ohne_t30 & an_schule]
kandidaten = campaign.dedupe_nearby(kandidaten, t30.ZUSAMMENFASSEN_M).reset_index(drop=True)

print(f"Kandidaten: {len(kandidaten)}")
kandidaten.plot(figsize=(6, 8), markersize=4)

## 5 · Bereits bestehende Aufgaben abziehen

Alles außer Fixed, Created und Skipped blockiert eine Neuanlage in 50 m. Außerdem fällt jedes
Zeichen weg, dessen ID Name einer erledigten Aufgabe ist.

Die offenen Aufgaben der Runde 2025 blockieren nicht: Sie sind nach dem Bild benannt statt nach
der Erkennung und verlinken oft ein altes Bild. Diese Runde ersetzt sie — Rebuild **mit**
„remove unmatched“.

In [ ]:
api_key = campaign.load_token(pfad_config, "API_KEY_MAPROULETTE")

# Everything but Fixed / Created / Skipped blocks a new task nearby: what was ticked
# off as "false positive" or "already fixed" must not come back.
bestehende = campaign.load_challenge_tasks(challenge_id, api_key)
print(f"blockierende Aufgaben in Challenge {challenge_id}: {len(bestehende)}")
kandidaten = campaign.drop_near_existing_tasks(kandidaten, bestehende, radius_m=t30.SUCHRADIUS_M)

# A sign whose id names a closed task would only update that task - it would stay closed.
kandidaten = campaign.drop_known_task_names(kandidaten, campaign.challenge_task_names(challenge_id, api_key))
print(f"neu anzulegen: {len(kandidaten)}")

## 6 · Neuestes Mapillary-Bild

Die Runde 2025 nahm `images[-1]` — das letzte Element einer Liste, die nicht nach
Aufnahmezeit sortiert ist — und schrieb Bild- und Erkennungs-ID vertauscht in die Aufgabe.
Hier kommt `captured_at` über die Graph-API gleich mit (`fields=id,images{id,captured_at}`),
und bis zu 50 Zeichen gehen in eine Anfrage.

In [ ]:
bilder = campaign.newest_image_ids(kandidaten["id"], campaign.load_token(pfad_config))

# Join on the string index: Mapillary ids exceed 2**53 and would be rounded as numbers.
kandidaten = kandidaten.set_index(kandidaten["id"].astype(str)).join(bilder).reset_index(drop=True)

kandidaten[["id", "last_seen_at", "dist_schule_m", "image_id", "image_captured_at"]].head()

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der das Zeichen erkannt wurde.
Stimmt die Bildauswahl, ist das verlinkte Bild ungefähr von diesem Tag.

In [ ]:
abstand_tage = (
    kandidaten["image_captured_at"].dt.tz_localize(None) - pd.to_datetime(kandidaten["last_seen_at"])
).dt.days.abs()

print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

if abstand_tage.notna().any():
    abstand_tage.plot(kind="hist", bins=40, figsize=(8, 3), title="Tage zwischen verlinktem Bild und last_seen_at")

## 7 · GeoJSON für MapRoulette schreiben

Die Datei behält ihren Namen, weil MapRoulette sie von GitHub holt. Ihr bisheriger Inhalt wird
**vor** dem Überschreiben eingelesen — sonst gäbe es im nächsten Schritt nichts zu vergleichen.

In [ ]:
vorheriger_stand = campaign.read_geojson(ziel_geojson)

aufgaben = campaign.build_maproulette_geojson(kandidaten, t30.aufgabe)
campaign.write_geojson(aufgaben, ziel_geojson)

if aufgaben["features"]:
    print(aufgaben["features"][0]["properties"]["instruction"])

## 8 · Vergleich mit dem vorherigen Stand

Stellt die Aufgaben-IDs des bisherigen Datei-Inhalts denen dieses Laufs gegenüber. Gegen die
Runde 2025 ist das wenig aussagekräftig: deren IDs waren Bild-IDs.

In [ ]:
if vorheriger_stand:
    campaign.print_comparison(campaign.compare_task_sets(vorheriger_stand, aufgaben))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

## 9 · In die Challenge hochladen

Direkt über die MapRoulette-API (`addFileTasks`), ohne Commit und ohne „Rebuild tasks“. Die
Aufgaben, die schon in der Challenge stehen, bleiben, wie sie sind. Danach prüft
`upload_tasks`, ob jede Aufgabe angekommen ist — MapRoulette meldet Erfolg auch dann, wenn
der Import intern scheitert.

Läuft nur mit `CAMPAIGN_UPLOAD=1`, und **noch gar nicht**: Seit maproulette-backend v4.11.0
(Oktober 2026) lehnt MapRoulette API-Keys ab; der angekündigte Ersatz, Personal Access Tokens,
fehlt noch. Bis dahin: Datei committen, pushen, „Rebuild tasks“.

In [ ]:
if hochladen:
    nachher = campaign.upload_tasks(
        challenge_id, aufgaben, api_key, data_origin_date=manifest["processed_date"][:10]
    )
    print(pd.Series(nachher).value_counts().to_string())
else:
    print(f"nicht hochgeladen ({len(aufgaben['features'])} Aufgaben) - dafür CAMPAIGN_UPLOAD=1 setzen")

---

## Challenge-Beschreibung (für MapRoulette)

## 🚸 Tempo-30 in Schul- und Kita-nähe anhand von Mapillary-Verkehrszeichen ergänzen (Deutschland)

Diese Challenge basiert auf automatisch erkannten **Tempo-30-Schildern** (DE:274-30) aus Mapillary-Bildern in Deutschland – fokussiert auf Bereiche **in der Nähe von Schulen und Kindergärten**.

### 📌 Kriterien für jede Aufgabe

Nur Aufgaben, die **alle** folgenden Bedingungen erfüllen, wurden berücksichtigt:

- Das Verkehrszeichen wurde **in Mapillary erkannt** (Tempo-30, ggf. mit Zusatzzeichen).
- Das Schild liegt **max. 450 m** von einer **Schule** oder einem **Kindergarten** entfernt (`amenity=school|kindergarten`).
- Im Umkreis von **50 m** um das Schild ist **keine Tempo-30-Erfassung in OSM** vorhanden
  *(einschließlich richtungs-, zeit- und fahrzeugspezifischer Varianten sowie Zonenangaben).*
- Das Schild wurde **mindestens 12 Monate** lang wiederholt gesichtet.
- Die **neueste Aufnahme stammt aus dem Jahr 2025 oder später**.
- Mehrere Schilder an derselben Stelle (bis 30 m Abstand) ergeben **eine** Aufgabe.

---

### 🔍 Was du tun solltest

1. Öffne den Ort in **Mapillary** (Bildserien prüfen) und einem passenden Viewer (z. B. **openstreetbrowser – maxspeed**).
   **Hinweis:** In **jeder Aufgabe** sind bereits **Direktlinks** an die richtige Position hinterlegt (Mapillary, openstreetbrowser – maxspeed, Unfallkarte).
2. **Prüfe, ob das erkannte Tempo-30-Schild tatsächlich (noch) existiert** – inkl. **Zusatzzeichen** (z. B. Zeitfenster, Schulweg).
   **Vergleiche anschließend die Situation mit der OSM-Erfassung an dieser Stelle:**
   Gibt es passende Tags (`maxspeed=*`, `maxspeed:*:conditional`, `maxspeed:forward/backward`, `maxspeed:hgv`, `zone:maxspeed=DE:30`)?
3. Ergänze ggf. die passenden OSM-Tags auf den betroffenen Straßenabschnitten:
   - **Konstant:** z. B. `maxspeed=30` (+ optional `source:maxspeed=sign`)
   - **Zeitabhängig (ohne Wochentage):** z. B. `maxspeed:conditional=30 @ (07:00-17:00)`
   - **Zeitabhängig (mit Wochentagen):** z. B. `maxspeed:conditional=30 @ (Mo-Fr 07:00-17:00)`
   - **Richtungsbezogen:** z. B. `maxspeed:forward=30` / `maxspeed:backward=30`
   - **Fahrzeugbezogen (Lkw/HGV):** z. B. `maxspeed:hgv=30` bzw. `maxspeed:hgv:conditional=30 @ (07:00-17:00)`
4. Wenn bereits alles korrekt gemappt ist, markiere die Aufgabe einfach **als erledigt**.

---

### 📚 Nützliche Links

- [**OSM-Wiki: maxspeed**](https://wiki.openstreetmap.org/wiki/DE:Key:maxspeed)
- [**OSM-Wiki: maxspeed:conditional**](https://wiki.openstreetmap.org/wiki/DE:Key:maxspeed)
- [**Traffic Sign Tool**](https://trafficsigns.osm-verkehrswende.org/DE?signs=DE:274-30,1042-33[Mo-Fr+16:00-18:00])

---

🗺️ Danke! Die erfassten Informationen können in Zukunft nützliche Erkenntnisse liefern, um **Schulwege sicherer zu machen**.

---

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```